In [11]:
!nvidia-smi

Mon Aug 31 04:36:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
!ls -la /content/drive/MyDrive/silent_speech
!ls -la /content/drive/MyDrive/silent_speech/output* 2>/dev/null

total 10543459
-rw------- 1 root root 6866471136 Aug 24 03:45 emg_dataset.h5
-rw------- 1 root root 3919507637 Aug 24 03:45 emg_data.tar.gz
drwx------ 2 root root       4096 Aug 24 03:49 KenLM
drwx------ 2 root root       4096 Aug 24 03:50 output
-rw------- 1 root root   10513993 Aug 31 04:28 resume.pt
total 113053
-rw------- 1 root root 10524837 Aug 24 04:00 model_20260824_035955_best.pt
-rw------- 1 root root 10524837 Aug 24 04:01 model_20260824_035955_last.pt
-rw------- 1 root root 10524837 Aug 24 04:28 model_20260824_040659_best.pt
-rw------- 1 root root 10524837 Aug 24 04:29 model_20260824_040659_last.pt
-rw------- 1 root root 10524837 Aug 24 04:47 model_20260824_044707_best.pt
-rw------- 1 root root 10524837 Aug 24 04:51 model_20260824_044707_last.pt
-rw------- 1 root root 10524837 Aug 24 05:08 model_20260824_045119_best.pt
-rw------- 1 root root 10524837 Aug 24 05:57 model_20260824_045119_last.pt
-rw------- 1 root root 10524837 Aug 24 06:11 model_20260824_061030_best.pt
-rw-----

In [14]:
%cd /content
!git clone https://github.com/MatteoFasulo/silent_speech.git
%cd /content/silent_speech
!git submodule update --init text_alignments
!tar -xzf text_alignments/text_alignments.tar.gz
!sed -i '/norm_layer=norm_layer,/d' architecture.py

/content
fatal: destination path 'silent_speech' already exists and is not an empty directory.
/content/silent_speech
fatal: detected dubious ownership in repository at '/content/silent_speech/text_alignments'
To add an exception for this directory, call:

	git config --global --add safe.directory /content/silent_speech/text_alignments
fatal: Unable to find current revision in submodule path 'text_alignments'


In [15]:
!pip install -q flashlight-text jiwer timm torchinfo torchprofile wandb tensorboard \
  librosa soundfile noisereduce resampy praat-textgrids unidecode \
  h5py scipy joblib matplotlib tqdm requests numpy

In [16]:
%env DATA_PATH=/content/data

env: DATA_PATH=/content/data


In [17]:
%cd /content/silent_speech
!mkdir -p /content/data/Gaddy/h5
# dataset file -> the exact local path the code reads
!cp /content/drive/MyDrive/silent_speech/emg_dataset.h5 /content/data/Gaddy/h5/ && echo "h5 copied" || echo "!! emg_dataset.h5 NOT on Drive"
# CTC decoder files
!cp -r /content/drive/MyDrive/silent_speech/KenLM /content/silent_speech/ && echo "KenLM copied" || echo "!! KenLM NOT on Drive"
# raw session folders: reuse tarball from Drive if present, else download fresh
!cp /content/drive/MyDrive/silent_speech/emg_data.tar.gz /content/data/Gaddy/ 2>/dev/null && echo "tar restored from Drive" || echo "no tar on Drive; downloading fresh (~several GB)"
!python download_data.py

/content/silent_speech
h5 copied
KenLM copied
tar restored from Drive
2026-08-31 04:44:37.786912: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-31 04:44:37.804490: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788151477.825662    5724 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788151477.832188    5724 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-31 04:44:37.853627: I tensorflow/core/platform/cpu_feature_guard.

In [18]:
import glob, os, re, json, torch
root = "/content/drive/MyDrive/silent_speech"

ckpts = sorted(glob.glob(f"{root}/**/model_*_last.pt", recursive=True), key=os.path.getmtime)
if not ckpts:
    ckpts = sorted(glob.glob(f"{root}/**/model_*_best.pt", recursive=True), key=os.path.getmtime)
assert ckpts, "no checkpoint (_last.pt / _best.pt) found on Drive"
print("checkpoints found (oldest -> newest):")
for c in ckpts: print("   ", c)
latest = ckpts[-1]                         # newest; paste a specific path here to pick another

raw = torch.load(latest, map_location="cpu", weights_only=False)
state = raw["state_dict"] if isinstance(raw, dict) and "state_dict" in raw else raw
idx = [int(re.match(r"blocks\.(\d+)\.", k).group(1)) for k in state if re.match(r"blocks\.(\d+)\.", k)]
n_layers = (max(idx) + 1) if idx else 4
torch.save({"state_dict": state}, f"{root}/resume.pt")

p = "/content/silent_speech/config/recognition_model.json"
cfg = json.load(open(p))
cfg["start_training_from"] = f"{root}/resume.pt"
cfg["num_layers"]          = n_layers
cfg["ckpt_directory"]      = os.path.dirname(latest)   # keep saving to the same folder
cfg["num_epochs"]          = 100
cfg["eval_interval"]       = 5
cfg["num_workers"]         = os.cpu_count()
json.dump(cfg, open(p, "w"), indent=4)
print(f"\nresuming from : {latest}\nnum_layers     : {n_layers}\nsaving to      : {cfg['ckpt_directory']}")

checkpoints found (oldest -> newest):
    /content/drive/MyDrive/silent_speech/output/model_20260824_035955_last.pt
    /content/drive/MyDrive/silent_speech/output/model_20260824_040659_last.pt
    /content/drive/MyDrive/silent_speech/output/model_20260824_044707_last.pt
    /content/drive/MyDrive/silent_speech/output/model_20260824_045119_last.pt
    /content/drive/MyDrive/silent_speech/output/model_20260824_061030_last.pt

resuming from : /content/drive/MyDrive/silent_speech/output/model_20260824_061030_last.pt
num_layers     : 4
saving to      : /content/drive/MyDrive/silent_speech/output


In [19]:
import os
def chk(label, path): print(f"{label:12}: {'OK' if os.path.exists(path) else 'MISSING <-- fix this'}")
chk("h5",         "/content/data/Gaddy/h5/emg_dataset.h5")
chk("raw silent", "/content/data/Gaddy/emg_data/silent_parallel_data")
chk("raw voiced", "/content/data/Gaddy/emg_data/voiced_parallel_data")
chk("KenLM lm",   "/content/silent_speech/KenLM/lm.bin")
chk("lexicon",    "/content/silent_speech/KenLM/gaddy_lexicon.txt")
chk("resume.pt",  "/content/drive/MyDrive/silent_speech/resume.pt")

h5          : OK
raw silent  : OK
raw voiced  : OK
KenLM lm    : OK
lexicon     : OK
resume.pt   : OK


In [20]:
%cd /content/silent_speech
!python recognition_model.py

/content/silent_speech
2026-08-31 04:46:30.999245: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-31 04:46:31.016964: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788151591.038144    6251 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788151591.044662    6251 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-31 04:46:31.066040: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to 

In [22]:
import os, glob
os.chdir("/content/silent_speech")
os.environ["BEST"] = sorted(glob.glob("/content/drive/MyDrive/silent_speech/**/model_*_best.pt", recursive=True), key=os.path.getmtime)[-1]
print("evaluating:", os.environ["BEST"])
!python recognition_model.py --evaluate_saved "$BEST"

evaluating: /content/drive/MyDrive/silent_speech/output/model_20260831_044634_best.pt
2026-08-31 05:15:00.052925: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-31 05:15:00.070797: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788153300.091882   14393 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788153300.098390   14393 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-31 05:15:00.119474: I tensorflow/core/platform/cp